In [0]:
from pyspark.sql.functions import hour, col, date_format, when, unix_timestamp
from pyspark.sql.types import *

In [0]:
df = spark.table("pysparkdbt_taxiproject.bronze.bronze_yellowtaxi")

## Derived Columns Extract time-based features from pickup timestamp for the gold layer grain (zone + day-of-week + hour) and compute trip duration for data-quality checks.

In [0]:
df = df.withColumn('pickup_hour', hour(col("tpep_pickup_datetime")))

## Feature Engineering — Derived Columns

In [0]:
df = df.withColumn("pickup_day_of_week", date_format(col("tpep_pickup_datetime"), "EEEE"))

##Derived Columns Extract pickup hour, day of week, weekend flag, and trip duration from the pickup/dropoff timestamps.

In [0]:
df = df.withColumn(
    "is_weekend",
    when(col("pickup_day_of_week").isin("Saturday", "Sunday"), True)
    .otherwise(False)
)

## Silver Layer — NYC Yellow Taxi Clean and validate bronze data.Reject bad rows to a quarantine table with reasons; derive time features for the gold grain.

In [0]:
df = df.withColumn("trip_duration",
        unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime")))

## Filtering the negative rows

In [0]:
df = df.withColumn(
    "reject_reason",
    when(~col("year_month").isin("2025-11", "2025-12"), "bad_timestamp")
    .when(col("trip_duration") < 0, "reversed_trip")
    .when(col("trip_duration") == 0, "zero_duration")
    .when(col("trip_distance") == 0, "zero_distance")
    .when(col("trip_distance") > 100, "distance_outlier")
    .when((col("fare_amount") < 0) | (col("total_amount") < 0), "negative_fare")
    .when(col("passenger_count") > 6, "invalid_passenger")
    .otherwise("good")
)

In [0]:
display(df.groupBy("reject_reason").count().orderBy("count", ascending=False))

reject_reason,count
good,7695991
negative_fare,411159
zero_distance,258731
zero_duration,118704
reversed_trip,1437
distance_outlier,397
bad_timestamp,24
invalid_passenger,7
